# LMP Vector Creation

**Notebook 03 in the Curtailed-2-Compute Analysis Workflow**

This notebook creates Locational Marginal Price (LMP) vectors for representative weeks, which are used in subsequent analyses to model electricity pricing patterns during high curtailment and high volatility periods.

## Overview

This notebook:
1. Loads and processes CAISO curtailment data
2. Integrates LMP data for NP15 pricing node
3. Identifies representative weeks (high curtailment, high volatility)
4. Creates 168-hour (weekly) vectors with curtailment and LMP data
5. Optionally adds carbon intensity data from WattTime API
6. Generates monthly representative vectors

## Workflow Position

- **Input**: Uses processed data from `01_curtailed_energy_analysis.ipynb` and TAC zone data from `02_TAC.ipynb`
- **Output**: Creates vector CSV files used in `04_electricity_costs_analysis.ipynb` and `05_scenario_financial_analysis.ipynb`
  - `vector_high_curtailment_week.csv` - Week with highest curtailment
  - `vector_high_volatility_week.csv` - Week with highest LMP volatility
  - `monthly_representative_vectors.xlsx` - Monthly representative vectors

## Key Features

- **Curtailment Data Processing**: Loads CAISO curtailment Excel files and processes 5-minute interval data
- **LMP Integration**: Combines curtailment data with Locational Marginal Price data for NP15 node
- **Representative Week Selection**: Identifies weeks with highest curtailment and highest price volatility
- **Vector Creation**: Generates complete 168-hour vectors (exactly one week) for analysis
- **Carbon Intensity**: Optionally fetches carbon intensity data from WattTime API

## Data Requirements

- CAISO curtailment Excel files in `../data/`
- LMP data files in `../data/LMP_Data/`
- WattTime API credentials (optional, for carbon intensity data)

## Output Files

- `vector_high_curtailment_week.csv` - 168-hour vector for highest curtailment week
- `vector_high_volatility_week.csv` - 168-hour vector for highest volatility week
- `monthly_representative_vectors.xlsx` - Monthly representative vectors with optional carbon data



## Data Requirements

### Source Data Files Needed

This notebook requires two types of source data:

#### 1. CAISO Curtailment Data (Excel files)
**Status**: Not included in repository - must be downloaded

**To obtain**:
1. Visit [CAISO Managing Oversupply Reports](https://www.caiso.com/informed/Pages/ManagingOversupply.aspx)
2. Download: `productionandcurtailmentsdata_YYYY.xlsx`
3. Place in: `../data/`

#### 2. CAISO LMP Data (CSV files)
**Status**: Not included in repository - must be downloaded

**To obtain**:
1. Visit [CAISO OASIS](http://oasis.caiso.com/)
2. Navigate to "Reports" → "Locational Marginal Price"
3. Select:
   - **Node**: `TH_NP15_GEN-APND` (Northern California pricing node)
   - **Time Period**: Monthly files for desired year (e.g., 2024)
   - **Data Items**: All LMP components
4. Download and save as: `YYYYMM_LMP.csv` (e.g., `202401_LMP.csv`)
5. Place in: `../data/LMP_Data/` directory

**Expected files for 2024 analysis**:
- `202401_LMP.csv` through `202412_LMP.csv` (12 monthly files)

**File size**: ~3,600-3,700 lines per month (hourly data)

### Reference Data Files (Included)

The following small reference files are included in the repository:
- `../data/reference/caiso_wind_solar_pcm.csv` - Used for NP15 proportion calculation

### Output Files Generated

This notebook generates:
- `vector_high_curtailment_week.csv` - Saved to `../data/processed/`
- `vector_high_volatility_week.csv` - Saved to `../data/processed/`
- `monthly_representative_vectors_with_carbon.xlsx` - Saved to notebook directory

### Carbon Intensity Data (WattTime API)

#### Why Carbon Intensity Matters

**Marginal Carbon Intensity** (also called CO2 MOER - Marginal Operating Emissions Rate) represents the carbon emissions (lbs CO2 per MWh) of the **marginal generator** that would be dispatched to meet an additional unit of electricity demand at a given time and location.

**Relevance to Curtailment Analysis**:
1. **Curtailment and Carbon**: When renewable energy is curtailed, the grid must rely more on fossil fuel generators to meet demand, increasing the marginal carbon intensity
2. **Time-of-Use Carbon Impact**: Carbon intensity varies significantly throughout the day, often peaking when solar generation declines (duck curve)
3. **Environmental Valuation**: Understanding carbon intensity during curtailment periods helps quantify the environmental cost of curtailment
4. **Data Center Siting**: For data centers considering renewable energy or carbon-neutral operations, understanding when curtailment occurs and its carbon implications is critical

#### How We Use It

- **Integration with Vectors**: Carbon intensity data is added to monthly representative vectors alongside LMP and curtailment data
- **Temporal Alignment**: Carbon data is fetched for the same time periods as our representative weeks (168-hour vectors)
- **Optional Feature**: Carbon data is optional - the notebook works without it, but provides enhanced analysis when available
- **Data Source**: WattTime API v3 provides historical marginal carbon intensity data for CAISO_NORTH region

#### Data Format

- **Units**: Pounds of CO2 per MWh (lbs CO2/MWh)
- **Temporal Resolution**: Hourly (resampled from WattTime's native resolution)
- **Coverage**: CAISO_NORTH region (Northern California)
- **Methodology**: Represents the marginal generator's emissions rate, not average grid emissions

**Note**: Carbon intensity data requires WattTime API credentials. The notebook will proceed without it if credentials are not available, but monthly vectors will not include carbon data.



In [ ]:
import pandas as pd
import os
import requests
from pathlib import Path
from requests.auth import HTTPBasicAuth

# --- Step 0: Define Constants and Configuration ---

# The directory where all your data files are located.
data_dir = Path("../data")
YEAR_TO_ANALYZE = 2024

In [ ]:
# The data-driven proportion for NP15 based on the 'caiso_wind_solar_pcm.csv' file.
# This represents the fraction of statewide renewable curtailment that occurs in NP15 zone.
NP15_RENEWABLE_PROPORTION = 0.40

# ============================================================================
# WattTime API Configuration for Carbon Intensity Data
# ============================================================================
# 
# Carbon intensity data provides the marginal CO2 emissions rate (lbs CO2/MWh)
# for the CAISO_NORTH region. This is useful for:
# - Understanding environmental impact of curtailment periods
# - Quantifying carbon costs during high curtailment hours
# - Supporting carbon-neutral data center siting decisions
#
# IMPORTANT: Set these as environment variables for security:
#   export WATTTIME_USERNAME="your_username"
#   export WATTTIME_PASSWORD="your_password"
# Or use a config file (not tracked in git) to store credentials
#
# The notebook will work without carbon data, but monthly vectors will not
# include the marginal_co2_lbs_per_mwh column.
# ============================================================================

import os
WATTTIME_USERNAME = os.getenv('WATTTIME_USERNAME', '')
WATTTIME_PASSWORD = os.getenv('WATTTIME_PASSWORD', '')
REGION = 'CAISO_NORTH'  # CAISO Northern California region

# Warn if credentials are not set
if not WATTTIME_USERNAME or not WATTTIME_PASSWORD:
    print("WARNING: WattTime API credentials not found in environment variables.")
    print("Carbon intensity data will not be available.")
    print("The notebook will proceed, but monthly vectors will not include carbon data.")
    print("\nTo enable carbon intensity data:")
    print("  export WATTTIME_USERNAME='your_username'")
    print("  export WATTTIME_PASSWORD='your_password'")
    print("\nFor more information, see: https://www.watttime.org/api-documentation/")

def login_to_watttime(username, password):
    """
    Logs into the WattTime API to get an authentication token.
    
    Note: According to WattTime API docs (https://docs.watttime.org/):
    - 403 Forbidden typically means the account is not registered, not activated, 
      or doesn't have subscription access
    - 401 Unauthorized means wrong credentials
    - Token expires after 30 minutes
    """
    url = 'https://api.watttime.org/login'  # Correct endpoint per API docs
    try:
        rsp = requests.get(url, auth=HTTPBasicAuth(username, password))
        
        # Handle specific error codes with helpful messages
        if rsp.status_code == 403:
            print(f"403 Forbidden: Account access denied.")
            print(f"  This usually means:")
            print(f"    1. Account not registered - use /register endpoint first")
            print(f"    2. Account not activated - check your email")
            print(f"    3. Account doesn't have subscription - historical data requires subscription")
            print(f"  See: https://docs.watttime.org/#tag/Authentication/operation/get_token_login_get")
            return None
        elif rsp.status_code == 401:
            print(f"401 Unauthorized: Invalid username or password")
            return None
        
        rsp.raise_for_status()
        token = rsp.json().get('token')
        if not token:
            print(f"Login failed. API Response: {rsp.text}")
            return None
        return token
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error during login: {e}")
        if hasattr(e.response, 'status_code'):
            print(f"  Status code: {e.response.status_code}")
            print(f"  Response: {e.response.text[:200]}")
        return None
    except Exception as e:
        print(f"An error occurred during login: {e}")
        return None

def fetch_v3_carbon_data(token, region, starttime, endtime):
    """
    Fetches historical marginal carbon data from the WattTime v3 API.
    
    Parameters:
    -----------
    token : str
        WattTime API authentication token
    region : str
        ISO region code (e.g., 'CAISO_NORTH')
    starttime : str
        Start time in ISO 8601 format (UTC)
    endtime : str
        End time in ISO 8601 format (UTC)
    
    Returns:
    --------
    dict or None
        JSON response from API containing carbon intensity data, or None if error
    
    Note:
    -----
    The API returns CO2 MOER (Marginal Operating Emissions Rate) which represents
    the carbon intensity (lbs CO2/MWh) of the marginal generator that would be
    dispatched to meet additional demand. This is different from average grid
    carbon intensity - it reflects the emissions impact of incremental load.
    """
    url = "https://api.watttime.org/v3/historical"
    headers = {"Authorization": f"Bearer {token}"}
    params = {
        "region": region,
        "start": starttime,
        "end": endtime,
        "signal_type": "co2_moer",  # Marginal Operating Emissions Rate
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error fetching carbon data: {e}")
        return None

def process_carbon_data_to_hourly(data):
    """
    Processes WattTime API response to hourly DataFrame.
    
    Parameters:
    -----------
    data : dict
        JSON response from WattTime API containing 'data' key with time series
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with hourly marginal carbon intensity (lbs CO2/MWh)
        Index: timestamp (UTC), Column: marginal_co2_lbs_per_mwh
    
    Note:
    -----
    Resamples to hourly by taking the mean of all data points within each hour.
    This aligns carbon data with our hourly LMP and curtailment vectors.
    """
    if not data or 'data' not in data or not data['data']:
        return pd.DataFrame()
    
    df = pd.DataFrame(data['data'])
    df.rename(columns={'point_time': 'timestamp', 'value': 'marginal_co2_lbs_per_mwh'}, inplace=True)
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df.set_index('timestamp', inplace=True)
    
    # Resample to hourly averages (aligns with LMP hourly data)
    hourly_df = df.resample('h').mean()
    return hourly_df[['marginal_co2_lbs_per_mwh']].sort_index()

In [ ]:
# Test WattTime API Credentials
# This cell tests whether the WattTime API credentials are valid and can fetch data

print("=" * 70)
print("WATTTIME API CREDENTIALS TEST")
print("=" * 70)

if not WATTTIME_USERNAME or not WATTTIME_PASSWORD:
    print("[WARN] Credentials not set in environment variables.")
    print("   Set WATTTIME_USERNAME and WATTTIME_PASSWORD to test API access.")
    print("   Example:")
    print("     export WATTTIME_USERNAME='your_username'")
    print("     export WATTTIME_PASSWORD='your_password'")
else:
    print(f"[OK] Username found: {WATTTIME_USERNAME[:3]}***")
    print(f"[OK] Password found: {'*' * len(WATTTIME_PASSWORD)}")
    print("\nTesting API authentication...")
    
    # Test 1: Login
    token = login_to_watttime(WATTTIME_USERNAME, WATTTIME_PASSWORD)
    if token:
        print(f"[OK] Authentication successful! Token received: {token[:20]}...")
        
        # Test 2: Fetch a small sample of carbon data
        print("\nTesting carbon data fetch...")
        from datetime import datetime, timedelta
        
        # Use a date from the past (1 week ago) as historical data is more reliably available
        # WattTime API typically has a delay before recent data is available
        end_time = datetime.utcnow() - timedelta(days=7)
        start_time = end_time - timedelta(days=1)  # 24 hours of data
        
        start_iso = start_time.strftime('%Y-%m-%dT%H:%M:%S+00:00')
        end_iso = end_time.strftime('%Y-%m-%dT%H:%M:%S+00:00')
        
        print(f"  Fetching data for: {start_iso} to {end_iso}")
        print(f"  Region: {REGION}")
        
        carbon_data = fetch_v3_carbon_data(token, REGION, start_iso, end_iso)
        
        if carbon_data and 'data' in carbon_data and carbon_data['data']:
            print(f"[OK] Carbon data fetch successful!")
            print(f"  Received {len(carbon_data['data'])} data points")
            
            # Test 3: Process the data
            print("\nTesting data processing...")
            carbon_df = process_carbon_data_to_hourly(carbon_data)
            
            if not carbon_df.empty:
                print(f"[OK] Data processing successful!")
                print(f"  Processed to {len(carbon_df)} hourly records")
                print(f"\nSample data:")
                print(carbon_df.head())
                print(f"\nStatistics:")
                print(f"  Mean: {carbon_df['marginal_co2_lbs_per_mwh'].mean():.2f} lbs CO2/MWh")
                print(f"  Min:  {carbon_df['marginal_co2_lbs_per_mwh'].min():.2f} lbs CO2/MWh")
                print(f"  Max:  {carbon_df['marginal_co2_lbs_per_mwh'].max():.2f} lbs CO2/MWh")
                print(f"\n[OK] All API tests passed! Carbon data integration is working correctly.")
            else:
                print("[WARN] Data processing returned empty DataFrame")
                print("  This may be normal if the time range has no data")
        else:
            print("[WARN] Carbon data fetch returned no data")
            print("  This may be normal if:")
            print("    - The time range is too recent (data may not be available yet)")
            print("    - The region code is incorrect")
            print("    - Your API account doesn't have access to historical data")
            print("\n  Try using a date from the past (e.g., 1 week ago) for testing")
    else:
        print("\n[FAIL] Authentication failed!")
        print("\n  Troubleshooting steps:")
        print("    1. Verify credentials are correct")
        print("    2. Check if account is registered:")
        print("       - Visit: https://docs.watttime.org/#tag/Authentication/operation/post_username_register_post")
        print("       - Use the /register endpoint if you haven't registered")
        print("    3. Check email for account activation")
        print("    4. Verify subscription status:")
        print("       - Historical data (/v3/historical) requires a subscription")
        print("       - Free accounts can preview CAISO_NORTH data")
        print("       - See: https://www.watttime.org/api-plans/")
        print("    5. Check API status: https://status.watttime.org/")
        print("    6. Contact support: support@WattTime.org")
        print("\n  For more information:")
        print("    - API Documentation: https://docs.watttime.org/")
        print("    - API Status: https://status.watttime.org/")

print("\n" + "=" * 70)


In [ ]:
def load_and_clean_curtailment(data_dir, year):
    """
    Loads and cleans the yearly curtailment Excel file.
    
    Searches for files in the configured data directory.
    """
    print(f"--- Processing Curtailment Data for {year} ---")
    
    # Try the configured data directory first, then the repository data directory
    possible_paths = [
        data_dir / f"productionandcurtailmentsdata_{year}.xlsx",
        Path("../data") / f"productionandcurtailmentsdata_{year}.xlsx"
    ]
    
    file_path = None
    for path in possible_paths:
        if path.exists():
            file_path = path
            break
    
    if file_path is None:
        print(f"Error: Could not find productionandcurtailmentsdata_{year}.xlsx")
        print(f"  Searched in: {[str(p.parent) for p in possible_paths]}")
        print(f"  Please download from: https://www.caiso.com/informed/Pages/ManagingOversupply.aspx")
        return pd.DataFrame()
    
    try:
        df = pd.read_excel(file_path, sheet_name="Curtailments")
        print(f"[OK] Successfully loaded {file_path.name} from {file_path.parent}")
        
        # Validate data
        if df.empty:
            print(f"Warning: {file_path.name} is empty")
            return pd.DataFrame()
        
        required_columns = ['Date', 'Hour', 'Interval', 'Solar Curtailment', 'Wind Curtailment']
        missing_cols = [col for col in required_columns if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Missing required columns in {file_path.name}: {missing_cols}")
            
    except Exception as e:
        print(f"Error loading {file_path.name}: {e}")
        return pd.DataFrame()

    # Build datetime column from separate Date/Hour/Interval columns
    df['timestamp'] = pd.to_datetime(df['Date']) + pd.to_timedelta(
        (df['Hour'] - 1) * 60 + (df['Interval'] - 1) * 5, unit='m'
    )
    
    # --- THIS IS THE CORRECTED LOGIC ---
    # Based on your reminder, we now correctly interpret the constructed timestamp 
    # as being in GMT/UTC from the start.
    df['timestamp'] = df['timestamp'].dt.tz_localize('UTC')
    
    # Select, rename, and clean core columns
    df = df[['timestamp', 'Solar Curtailment', 'Wind Curtailment']].copy()
    df.rename(columns={
        'Solar Curtailment': 'Solar_Curtailment_CAISO_MW',
        'Wind Curtailment': 'Wind_Curtailment_CAISO_MW'
    }, inplace=True)
    df.set_index('timestamp', inplace=True)
    # Fill missing values: NaN curtailment values should be treated as zero (no curtailment)
    df = df.fillna(0)

    # Handle duplicate timestamps: if any exist, take the mean (not sum) to avoid double-counting
    # Curtailment values represent instantaneous power, so duplicates should be averaged, not summed
    if df.index.duplicated().any():
        print(f"Warning: Found {df.index.duplicated().sum()} duplicate timestamps. Taking mean values.")
        df = df.groupby(df.index).mean()
    
    return df

def load_and_clean_lmp(data_dir, year, node_id='TH_NP15_GEN-APND'):
    """
    Loads and combines all monthly LMP CSV files for a given year.
    
    Searches in:
    1. data_dir/LMP_Data/ (primary location)
    2. data_dir/ (fallback)
    3. ../data/LMP_Data/ (alternative path)
    """
    print(f"\n--- Processing LMP Data for {year} ---")
    
    # Try multiple possible locations for LMP data
    search_paths = [
        data_dir / "LMP_Data",  # Primary location
        Path("../data/LMP_Data"),  # Alternative path
        data_dir,  # Root data directory
        Path("../data")  # Alternative root
    ]
    
    all_files = []
    for search_path in search_paths:
        if search_path.exists():
            files = list(search_path.glob(f"{year}*_LMP.csv"))
            if files:
                all_files.extend(files)
                print(f"Found {len(files)} files in {search_path}")
                break
    
    if not all_files:
        print("=" * 70)
        print(f"ERROR: No LMP files found for year {year}")
        print("=" * 70)
        print(f"Searched in: {[str(p) for p in search_paths]}")
        print(f"Expected pattern: {year}*_LMP.csv (e.g., {year}01_LMP.csv)")
        print("\nTo obtain LMP data:")
        print("1. Visit: http://oasis.caiso.com/")
        print("2. Navigate to: Reports → Locational Marginal Price")
        print("3. Select:")
        print(f"   - Node: {node_id}")
        print(f"   - Time Period: Monthly files for {year}")
        print("   - Data Items: All LMP components")
        print(f"4. Download files as: {year}MM_LMP.csv (e.g., {year}01_LMP.csv)")
        print(f"5. Place files in: {data_dir}/LMP_Data/")
        print("=" * 70)
        return pd.DataFrame()
        
    df_list = []
    for file in all_files:
        print(f"Processing file: {file.name}")
        try:
            df = pd.read_csv(file)
            
            # Validate required columns
            required_cols = ['NODE_ID', 'INTERVALSTARTTIME_GMT', 'XML_DATA_ITEM', 'MW']
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                print(f"  Warning: Missing columns in {file.name}: {missing_cols}")
                continue
            
            # Filter for the specific node
            node_data = df[df['NODE_ID'] == node_id].copy()
            if node_data.empty:
                print(f"  Warning: No data for node {node_id} in {file.name}")
                continue
            
            df_pivot = node_data.pivot(index='INTERVALSTARTTIME_GMT', columns='XML_DATA_ITEM', values='MW')
            df_pivot.reset_index(inplace=True)
            df_list.append(df_pivot)
        except Exception as e:
            print(f"  Error processing {file.name}: {e}")
            continue
        
    if not df_list:
        print("Warning: No valid LMP data files were processed")
        return pd.DataFrame()
    
    final_df = pd.concat(df_list, ignore_index=True)
    
    # Validate that LMP_PRC column exists
    if 'LMP_PRC' not in final_df.columns:
        print("Error: LMP_PRC column not found in LMP data. Check data format.")
        return pd.DataFrame()
    
    final_df.rename(columns={'LMP_PRC': 'LMP_NP15', 'INTERVALSTARTTIME_GMT': 'timestamp'}, inplace=True)
    
    # This logic was already correct: convert the GMT/UTC timestamp string to a proper UTC datetime object
    final_df['timestamp'] = pd.to_datetime(final_df['timestamp'], utc=True)
    final_df.set_index('timestamp', inplace=True)

    # Handle duplicate timestamps: if any exist, take the mean (not sum) to avoid double-counting
    # LMP values represent prices at specific times, so duplicates should be averaged
    if final_df.index.duplicated().any():
        print(f"Warning: Found {final_df.index.duplicated().sum()} duplicate timestamps in LMP data. Taking mean values.")
        final_df = final_df.groupby(final_df.index).mean()
    
    print(f"[OK] Successfully processed {len(df_list)} LMP files")
    print(f"  Total records: {len(final_df)}")
    
    return final_df[['LMP_NP15']]

In [ ]:
# Load the datasets using the functions
curtailment_statewide_df = load_and_clean_curtailment(data_dir, YEAR_TO_ANALYZE)
lmp_np15_df = load_and_clean_lmp(data_dir, YEAR_TO_ANALYZE)

# Validate data loading
if curtailment_statewide_df.empty:
    raise ValueError(f"Failed to load curtailment data for year {YEAR_TO_ANALYZE}. Check file paths.")
if lmp_np15_df.empty:
    raise ValueError(f"Failed to load LMP data for year {YEAR_TO_ANALYZE}. Check file paths and node ID.")

# Calculate the estimated NP15 curtailment
# Note: CAISO reports statewide curtailment. We estimate NP15 zone curtailment using
# a data-driven proportion (0.40) based on renewable capacity mix in NP15 vs. statewide.
# This is a simplification; actual zonal curtailment would require detailed generator-level data.
curtailment_statewide_df = curtailment_statewide_df.copy()
curtailment_statewide_df['Total_Curtailment_CAISO_MW'] = (
    curtailment_statewide_df['Solar_Curtailment_CAISO_MW'] + 
    curtailment_statewide_df['Wind_Curtailment_CAISO_MW']
)
# Apply the proportion to estimate NP15 curtailment
curtailment_statewide_df['Total_Curtailment_NP15_MW'] = (
    curtailment_statewide_df['Total_Curtailment_CAISO_MW'] * NP15_RENEWABLE_PROPORTION
)
np15_curtailment_df = curtailment_statewide_df[['Total_Curtailment_NP15_MW']].copy()

print("\n--- Zonal Curtailment Calculated ---")
print(f"NP15 Renewable Proportion: {NP15_RENEWABLE_PROPORTION:.1%}")
print(f"Total statewide curtailment (sample): {curtailment_statewide_df['Total_Curtailment_CAISO_MW'].sum():.2f} MW")
print(f"Estimated NP15 curtailment (sample): {np15_curtailment_df['Total_Curtailment_NP15_MW'].sum():.2f} MW")
display(np15_curtailment_df.head())

if not lmp_np15_df.empty:
    print("\n--- NP15 LMP Data Loaded ---")
    print(f"LMP data range: {lmp_np15_df.index.min()} to {lmp_np15_df.index.max()}")
    print(f"Number of LMP records: {len(lmp_np15_df)}")
    display(lmp_np15_df.head())

In [ ]:
# Merge curtailment and LMP data
# Use 'left' join to keep all LMP hours (LMP data is the primary time series)
# Missing curtailment values are filled with 0 (no curtailment during those hours)
print("\n--- Merging Datasets ---")

# Resample curtailment data to hourly if needed (LMP is hourly, curtailment is 5-minute)
# Take the mean of curtailment values within each hour
if np15_curtailment_df.index.freq is None:
    # Resample 5-minute curtailment data to hourly by taking mean
    np15_curtailment_hourly = np15_curtailment_df.resample('h').mean()
else:
    np15_curtailment_hourly = np15_curtailment_df

# Join datasets: left join preserves all LMP timestamps
final_df = lmp_np15_df.join(np15_curtailment_hourly, how='left')
final_df['Total_Curtailment_NP15_MW'] = final_df['Total_Curtailment_NP15_MW'].fillna(0)

# Sort timestamps so later nearest-neighbor lookups are valid.
final_df = final_df.sort_index()

# Validate: check for duplicate timestamps
if final_df.index.duplicated().any():
    print(f"Warning: Found duplicate timestamps after merge. Removing duplicates (keeping first).")
    final_df = final_df[~final_df.index.duplicated(keep='first')]
    final_df = final_df.sort_index()

# Data quality checks
print(f"Merge successful. Final DataFrame contains {len(final_df)} rows.")
print(f"Time range: {final_df.index.min()} to {final_df.index.max()}")
print(f"Hours with curtailment > 0: {(final_df['Total_Curtailment_NP15_MW'] > 0).sum()}")
print(f"Hours with zero curtailment: {(final_df['Total_Curtailment_NP15_MW'] == 0).sum()}")
print(f"Missing LMP values: {final_df['LMP_NP15'].isna().sum()}")

# Validate data quality
if final_df['LMP_NP15'].isna().any():
    print("Warning: Some LMP values are missing. Consider investigating data quality.")
if len(final_df) < 8000:  # Expect ~8760 hours per year
    print(f"Warning: Expected ~8760 hours per year, but found only {len(final_df)} records.")

display(final_df.head())
display(final_df.describe())

In [ ]:
if 'final_df' in locals() and not final_df.empty:
    print("\n--- Identifying Representative Weeks ---")
    
    # Data is already hourly, so we can directly resample by week
    # Note: Converting MW to MWh for weekly aggregation (MW × hours = MWh)
    hourly_df = final_df.copy()
    
    # Resample by week (7-day periods) and calculate statistics
    # Total_Curtailment_MWh: Sum of hourly curtailment (MW) = total energy curtailed in MWh
    # Avg_LMP: Mean price during the week
    # LMP_Volatility: Standard deviation of prices (measures price variability)
    weekly_stats = hourly_df.resample('7D', origin='start').agg({
        'Total_Curtailment_NP15_MW': 'sum',  # Sum MW to get total MWh for the week
        'LMP_NP15': ['mean', 'std']  # Mean and standard deviation of LMP
    })
    
    # Flatten column names
    weekly_stats.columns = ['Total_Curtailment_MWh', 'Avg_LMP', 'LMP_Volatility']
    
    # Find representative weeks
    # High curtailment week: week with maximum total curtailment (energy not delivered)
    # High volatility week: week with maximum price volatility (std deviation)
    weeks_to_extract = {
        'high_curtailment': weekly_stats['Total_Curtailment_MWh'].idxmax(),
        'high_volatility': weekly_stats['LMP_Volatility'].idxmax()
    }
    
    print(f"Highest Curtailment Week starts on: {weeks_to_extract['high_curtailment']}")
    print(f"  Total curtailment: {weekly_stats.loc[weeks_to_extract['high_curtailment'], 'Total_Curtailment_MWh']:.2f} MWh")
    print(f"  Average LMP: ${weekly_stats.loc[weeks_to_extract['high_curtailment'], 'Avg_LMP']:.2f}/MWh")
    
    print(f"\nHighest LMP Volatility Week starts on: {weeks_to_extract['high_volatility']}")
    print(f"  LMP volatility (std dev): ${weekly_stats.loc[weeks_to_extract['high_volatility'], 'LMP_Volatility']:.2f}/MWh")
    print(f"  Average LMP: ${weekly_stats.loc[weeks_to_extract['high_volatility'], 'Avg_LMP']:.2f}/MWh")
    
    print("\nTop 5 weeks by total curtailment:")
    display(weekly_stats.sort_values(by='Total_Curtailment_MWh', ascending=False).head())
else:
    print("Error: final_df not available. Check data loading steps above.")

In [ ]:
# Extract and save complete 168-hour weekly vectors
if 'weeks_to_extract' in locals():
    print("\n--- Extracting Complete 168-Hour Vectors ---")
    for name, start_date in weeks_to_extract.items():
        # Create exact 168-hour range
        hour_range = pd.date_range(start=start_date, periods=168, freq='h')
        
        # Extract data for these hours, fill missing with interpolation
        weekly_data = []
        for hour in hour_range:
            if hour in final_df.index:
                weekly_data.append(final_df.loc[hour])
            else:
                # Fill missing hours with 0 curtailment and nearest LMP
                nearest_idx = final_df.index.get_indexer([hour], method='nearest')[0]
                nearest_lmp = final_df['LMP_NP15'].iloc[nearest_idx]
                weekly_data.append(pd.Series({'Total_Curtailment_NP15_MW': 0, 'LMP_NP15': nearest_lmp}))
        
        weekly_vector = pd.DataFrame(weekly_data, index=hour_range)
        weekly_vector.index = weekly_vector.index.strftime('%Y-%m-%d %H:%M')
        weekly_vector.index.name = 'datetime'
        
        # Save to processed data directory
        output_dir = Path("../data/processed")
        output_dir.mkdir(parents=True, exist_ok=True)
        filename = output_dir / f"vector_{name}_week.csv"
        weekly_vector.to_csv(filename)
        print(f"-> Saved {filename} (exactly 168 hours)")

In [ ]:
# =============================================================================
# Figure: Average Hourly LMP and Components (The Duck Curve)
# =============================================================================
# This cell creates the hourly LMP pattern showing energy and congestion components
# Note: We need to reload LMP data with all components (not just total LMP)

import matplotlib.pyplot as plt

# Reload LMP data with components
lmp_files_path = Path("../data/LMP_Data")
if not lmp_files_path.exists():
    lmp_files_path = data_dir / "LMP_Data"

all_lmp_files = list(lmp_files_path.glob(f"{YEAR_TO_ANALYZE}*_LMP.csv"))

if all_lmp_files:
    print(f"Loading LMP components from {len(all_lmp_files)} files...")
    
    df_components_list = []
    for file in all_lmp_files:
        df = pd.read_csv(file)
        node_data = df[df['NODE_ID'] == 'TH_NP15_GEN-APND'].copy()
        if not node_data.empty:
            df_pivot = node_data.pivot(index='INTERVALSTARTTIME_GMT', columns='XML_DATA_ITEM', values='MW')
            df_pivot.reset_index(inplace=True)
            df_components_list.append(df_pivot)
    
    if df_components_list:
        lmp_components_df = pd.concat(df_components_list, ignore_index=True)
        lmp_components_df['timestamp'] = pd.to_datetime(lmp_components_df['INTERVALSTARTTIME_GMT'], utc=True)
        lmp_components_df.set_index('timestamp', inplace=True)
        
        # Rename columns for clarity
        lmp_components_df.rename(columns={
            'LMP_PRC': 'total_lmp',
            'LMP_ENE_PRC': 'lmp_energy',
            'LMP_CONG_PRC': 'lmp_congestion',
            'LMP_LOSS_PRC': 'lmp_losses',
            'LMP_GHG_PRC': 'lmp_ghg'
        }, inplace=True)
        
        # Convert UTC to Pacific Time for proper duck curve alignment
        # California operates on Pacific Time, so we need local hours
        lmp_components_df_pacific = lmp_components_df.copy()
        lmp_components_df_pacific.index = lmp_components_df_pacific.index.tz_convert('America/Los_Angeles')
        
        # Calculate hourly averages by PACIFIC hour of day
        hourly_avg = lmp_components_df_pacific[['lmp_energy', 'lmp_congestion', 'total_lmp']].groupby(
            lmp_components_df_pacific.index.hour
        ).mean()
        
        print(f"Converted {len(lmp_components_df)} hours from UTC to Pacific Time")
        
        # Plot the duck curve
        plt.figure(figsize=(12, 7))
        plt.plot(hourly_avg.index, hourly_avg['lmp_energy'], label='Energy Component', linestyle='--', color='blue')
        plt.plot(hourly_avg.index, hourly_avg['lmp_congestion'], label='Congestion Component', linestyle='--', color='red')
        plt.plot(hourly_avg.index, hourly_avg['total_lmp'], label='Total LMP', color='green', linewidth=2)
        plt.title('Average Hourly LMP and Components (The Duck Curve)')
        plt.xlabel('Hour of the Day')
        plt.ylabel('Price ($/MWh)')
        plt.xticks(range(24))
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()
        
        # Export the data
        hourly_avg.index.name = 'Hour'
        hourly_avg.to_csv('avg_hourly_lmp_components_duck_curve.csv')
        print("[OK] Exported: avg_hourly_lmp_components_duck_curve.csv")
        print(f"\nData summary:")
        print(hourly_avg.describe())
else:
    print("No LMP files found. Cannot create duck curve figure.")

In [ ]:
# Create Monthly Representative Vectors with Carbon Data
# 
# This section creates monthly representative vectors that include:
# 1. LMP data (pricing)
# 2. Curtailment data (renewable energy not delivered)
# 3. Carbon intensity data (marginal CO2 emissions rate) - OPTIONAL
#
# Carbon intensity data helps quantify the environmental impact of curtailment:
# - During curtailment periods, the grid relies more on fossil fuel generators
# - This increases the marginal carbon intensity (more CO2 per MWh)
# - Understanding this relationship is important for carbon-neutral operations
#
if 'final_df' in locals():
    print("\n--- Creating Monthly Representative Vectors with Carbon Data ---")
    print("Note: Carbon intensity data is optional. The notebook will proceed without it")
    print("      if WattTime API credentials are not available.\n")
    
    # Login to WattTime API
    watttime_token = login_to_watttime(WATTTIME_USERNAME, WATTTIME_PASSWORD)
    if not watttime_token:
        print("[WARN] Failed to authenticate with WattTime API. Proceeding without carbon data.")
        print("  Monthly vectors will not include marginal_co2_lbs_per_mwh column.")
    else:
        print("[OK] WattTime API authenticated. Carbon intensity data will be included.")
    
    # Monthly selection logic: week with highest curtailment in each month
    monthly_vectors = {}
    
    for month in range(1, 13):
        month_data = final_df[final_df.index.month == month]
        if len(month_data) == 0:
            continue
            
        # Find week with highest curtailment in this month
        monthly_weekly_stats = month_data.resample('7D', origin='start').agg(
            Total_Curtailment_MWh=('Total_Curtailment_NP15_MW', 'sum'),
            Avg_LMP=('LMP_NP15', 'mean'),
            LMP_Volatility=('LMP_NP15', 'std')
        )
        
        if len(monthly_weekly_stats) > 0:
            best_week_start = monthly_weekly_stats['Total_Curtailment_MWh'].idxmax()
            
            # Fetch carbon data for this week if token available
            carbon_df = pd.DataFrame()
            if watttime_token:
                end_date = best_week_start + pd.Timedelta(days=7)
                start_iso = best_week_start.strftime('%Y-%m-%dT%H:%M:%S+00:00')
                end_iso = end_date.strftime('%Y-%m-%dT%H:%M:%S+00:00')
                
                carbon_data = fetch_v3_carbon_data(watttime_token, REGION, start_iso, end_iso)
                if carbon_data:
                    carbon_df = process_carbon_data_to_hourly(carbon_data)
            
            # Create 168-hour vector for this month
            hour_range = pd.date_range(start=best_week_start, periods=168, freq='h')
            
            monthly_data = []
            for hour in hour_range:
                row_data = {}
                
                if hour in final_df.index:
                    row_data.update(final_df.loc[hour].to_dict())
                else:
                    nearest_idx = final_df.index.get_indexer([hour], method='nearest')[0]
                    nearest_lmp = final_df['LMP_NP15'].iloc[nearest_idx]
                    row_data.update({'Total_Curtailment_NP15_MW': 0, 'LMP_NP15': nearest_lmp})
                
                # Add carbon data if available
                # Only add carbon column if we have carbon data for this month
                if not carbon_df.empty:
                    # Ensure timezone alignment: normalize both to UTC for comparison
                    hour_utc = pd.Timestamp(hour).tz_localize('UTC') if hour.tz is None else hour.tz_convert('UTC')
                    
                    # Try exact match first
                    if hour_utc in carbon_df.index:
                        row_data['marginal_co2_lbs_per_mwh'] = carbon_df.loc[hour_utc, 'marginal_co2_lbs_per_mwh']
                    else:
                        # Use nearest neighbor if exact match not found
                        try:
                            nearest_idx = carbon_df.index.get_indexer([hour_utc], method='nearest')[0]
                            if nearest_idx >= 0:
                                # Check if nearest is within 1 hour (to avoid matching wrong days)
                                time_diff = abs((carbon_df.index[nearest_idx] - hour_utc).total_seconds())
                                if time_diff <= 3600:  # Within 1 hour
                                    row_data['marginal_co2_lbs_per_mwh'] = carbon_df.iloc[nearest_idx]['marginal_co2_lbs_per_mwh']
                                else:
                                    row_data['marginal_co2_lbs_per_mwh'] = None
                            else:
                                row_data['marginal_co2_lbs_per_mwh'] = None
                        except Exception as e:
                            # If matching fails, set to None
                            row_data['marginal_co2_lbs_per_mwh'] = None
                # If carbon_df is empty (no API access or no data), don't add the column
                # The column will only be added if at least one month has carbon data
                
                monthly_data.append(pd.Series(row_data))
            
            monthly_vector = pd.DataFrame(monthly_data, index=hour_range)
            monthly_vector.index = monthly_vector.index.strftime('%Y-%m-%d %H:%M')
            monthly_vector.index.name = 'datetime'
            
            # If carbon data was successfully added, note it
            if not carbon_df.empty and 'marginal_co2_lbs_per_mwh' in monthly_vector.columns:
                carbon_count = monthly_vector['marginal_co2_lbs_per_mwh'].notna().sum()
                if carbon_count > 0:
                    print(f"Created vector for Month {month:02d}: {best_week_start.strftime('%Y-%m-%d')} (168 hours, {carbon_count} hours with carbon data)")
                else:
                    print(f"Created vector for Month {month:02d}: {best_week_start.strftime('%Y-%m-%d')} (168 hours, carbon data fetched but no matches)")
            else:
                print(f"Created vector for Month {month:02d}: {best_week_start.strftime('%Y-%m-%d')} (168 hours, no carbon data)")
            
            monthly_vectors[f'Month_{month:02d}'] = monthly_vector
    
    # Save all monthly vectors to a single Excel file with multiple sheets
    if monthly_vectors:
        # Check if any vectors have carbon data
        has_carbon_data = any('marginal_co2_lbs_per_mwh' in df.columns and df['marginal_co2_lbs_per_mwh'].notna().any() 
                             for df in monthly_vectors.values())
        
        with pd.ExcelWriter('monthly_representative_vectors_with_carbon.xlsx', engine='openpyxl') as writer:
            for sheet_name, vector_df in monthly_vectors.items():
                vector_df.to_excel(writer, sheet_name=sheet_name)
        
        print(f"\n-> Saved monthly_representative_vectors_with_carbon.xlsx with {len(monthly_vectors)} sheets")
        if has_carbon_data:
            print("Each sheet contains 168 hours with LMP, curtailment, and carbon intensity data")
        else:
            print("Each sheet contains 168 hours with LMP and curtailment data (carbon data not available)")
            print("Note: To include carbon data, set WATTTIME_USERNAME and WATTTIME_PASSWORD environment variables")
            print("      and ensure your WattTime API account has access to historical data")
        
        # Display sample from first month
        first_month = list(monthly_vectors.keys())[0]
        print(f"\nSample from {first_month}:")
        display(monthly_vectors[first_month].head())
        print(f"Shape: {monthly_vectors[first_month].shape}")
        
        # Show carbon data statistics if available
        if has_carbon_data:
            first_df = monthly_vectors[first_month]
            if 'marginal_co2_lbs_per_mwh' in first_df.columns:
                carbon_non_null = first_df['marginal_co2_lbs_per_mwh'].notna().sum()
                print(f"\nCarbon data coverage in {first_month}: {carbon_non_null}/168 hours ({carbon_non_null/168:.1%})")
                if carbon_non_null > 0:
                    print(f"  Mean carbon intensity: {first_df['marginal_co2_lbs_per_mwh'].mean():.2f} lbs CO2/MWh")
                    print(f"  Range: {first_df['marginal_co2_lbs_per_mwh'].min():.2f} - {first_df['marginal_co2_lbs_per_mwh'].max():.2f} lbs CO2/MWh")